# 2WikiMultiHopQA ở n=1500 - khép khẳng định duy nhất n=500 không đủ

2Wiki có **12,576** câu; mình đánh giá trên 500. Câu hỏi đúng phải hỏi không
phải "500 trên 12,576 có ít quá không" mà "500 đủ cho *khẳng định nào*".

Ở n=500 ngưỡng phát hiện là **8.1 F1** (bootstrap ghép cặp, power 0.8). Đối
chiếu từng khẳng định trên 2Wiki:

| khẳng định | hiệu | CI | n cần |
|---|---|---|---|
| IterCOMP − LLMLingua-2 | **+16.84** | [+12.36, +21.50] | 37 |
| Oracle − Raw | **+20.57** | [+16.33, +24.87] | 22 |
| IterCOMP − Raw | −2.69 | [−7.03, **+1.67**] | **1,304** |

Hai dòng đầu - gồm **khẳng định trung tâm của paper** - đã sạch với n dư 13×
và 23×. Chạy hết 12,576 câu không mua thêm gì cho chúng, mà tốn **25.6h** T4:
vượt quota tuần và không nhét được vào một session 9h.

Chỉ **một** khẳng định là 500 không đủ: `IterCOMP − Raw`. CI của nó cắt qua 0
nên báo cáo đang phải ghi "chưa kết luận", và đây lại đúng là chỗ luận điểm
*reader-dependence* cần - trên HotpotQA cùng hiệu đó là −7.99, CI sạch.

Kernel này chạy 2Wiki ở **n=1500** (dư trên mức 1,304 cần) để hiệu đó hoặc
sạch hoặc được chứng minh là thật sự bằng 0.

| | |
|---|---|
| $k$ | **85** cho 2Wiki - đúng §5.1 của paper |
| Chi phí | 1500/500 × 61 phút = **~3.05h** T4 |
| Ghi ra | `results/2wiki_1500_7b.json` - file n=500 giữ nguyên để truy nguồn |

**Ba kết cục, cả ba đều dùng được**

- CI sạch phía âm → Raw thắng IterCOMP trên 2Wiki *có ý nghĩa*, khớp HotpotQA,
  luận điểm reader-dependence mạnh lên trên 2/3 dataset tiếng Anh.
- CI vẫn cắt 0 ở n=1500 → hiệu thật sự nhỏ; "chưa kết luận" đổi thành
  "bằng 0 trong khoảng ±2", là một phát biểu chắc chứ không phải thiếu mẫu.
- CI sạch phía dương → IterCOMP thắng Raw trên 2Wiki; phải sửa báo cáo, và
  n=500 đã dẫn mình sai - chính vì vậy mới chạy.


In [ ]:
# ══ CỬA CHẶN: GPU có chạy được bitsandbytes 4-bit không? ══
# Kaggle cấp NGẪU NHIÊN P100 (sm_60) hoặc T4 (sm_75) nếu metadata không ghi rõ
# machine_shape. PyTorch của Kaggle chỉ build cho sm_70 trở lên, nên trên P100
# bitsandbytes chết bằng SIGSEGV giữa lúc nạp trọng số:
#
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
#     rc=-11
#
# Lỗi đó mất ~2 phút mới hiện và thông báo không nói gì về nguyên nhân. Ô này
# phát hiện trong 5 giây và nói thẳng phải làm gì.
import torch

assert torch.cuda.is_available(), (
    'Không có GPU. Settings > Accelerator > GPU T4 x2, rồi chạy lại.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  sm_{major}{minor}  {gb:.1f} GB')

if major < 7:
    raise SystemExit(
        f'\n{name} là sm_{major}{minor} - PyTorch của Kaggle không hỗ trợ.\n'
        'Cần T4 (sm_75). Hai cách:\n'
        '  1. Settings > Accelerator > chọn "GPU T4 x2" (không phải "GPU P100")\n'
        '  2. Nếu đẩy bằng CLI: thêm "machine_shape": "NvidiaTeslaT4" vào\n'
        '     kernel-metadata.json\n'
        'Chạy tiếp trên P100 sẽ SIGSEGV lúc nạp mô hình 4-bit.')

print('✓ GPU chạy được bitsandbytes 4-bit')

In [ ]:
# ══ GHIM transformers VỀ 4.x - ĐÂY LÀ BẢN VÁ THẬT ══
# Kaggle nay ship transformers 5.0.0. llmlingua 0.2.2 viết cho 4.x, và hai bên
# đòi hai thứ LOẠI TRỪ NHAU cho cùng một object `past_key_values`:
#
#   transformers 5.0  ->  phải là Cache, gọi .get_seq_length()
#   llmlingua 0.2.2   ->  phải là list, lặp `for k, v in past_key_values`
#
# Không monkey-patch nào làm hài lòng cả hai, vì cùng một object đi qua cả hai
# nơi. SÁU lần vá đều chết vì cố làm điều bất khả. Đường LongLLMLingua đã được
# chạy thử THÀNH CÔNG trên 4.45.2 ở máy dev, nên ghim về đúng bản đó cũng khép
# luôn lỗ hổng "chạy được ở đây, chết trên Kaggle".
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. Nếu transformers đã '
    f'được import trước khi pip chạy thì phải Restart & Run All.')


In [ ]:
# Lấy mã nguồn từ gist. Repo là private nên Colab không clone ẩn danh được;
# gist thì public, tải thẳng, không phải upload tay lần nào. URL không kèm SHA
# nên luôn trỏ bản mới nhất.
import os, sys, base64, io, zipfile, urllib.request, shutil

# Thư mục làm việc: Colab dùng /content, Kaggle dùng /kaggle/working. Phát hiện
# thay vì cứng hoá - cùng notebook chạy được cả hai, và trên máy khác thì lùi về
# thư mục hiện tại.
BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'repo')
SRC_URL = ('https://gist.githubusercontent.com/nhantrnh/'
           'aeec3610fb9541b5f8b388cafd424eb9/raw/itercomp_src_b64.txt')
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data',
        'metrics', 'fertility', 'stats', 'budget']

def missing_in(root):
    return [m for m in NEED
            if not os.path.exists(os.path.join(root, 'src', 'itercomp', m + '.py'))]

# Ra khỏi REPO trước khi xoá: nếu lần chạy trước đã os.chdir(REPO) thì cwd đang
# nằm trong thư mục sắp bị xoá, và mọi thao tác mở file sau đó sẽ ném
# FileNotFoundError khó hiểu.
os.chdir(BASE)

# Xoá sạch: extractall KHÔNG tự xoá file cũ, nên một gói cũ có thể "sống sót"
# qua nhiều lần chạy và che mất bản mới.
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]          # bỏ module đã nạp trong bộ nhớ
os.makedirs(REPO, exist_ok=True)

with urllib.request.urlopen(SRC_URL, timeout=60) as r:
    blob = base64.b64decode(r.read())
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(REPO)
print(f'✓ tải mã nguồn từ gist ({len(blob)/1024:.0f} KB)')

miss = missing_in(REPO)
if miss:
    raise SystemExit(f"Gói mã nguồn thiếu module: {', '.join(miss)}")

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

from itercomp import itercomp, load_dataset, normalize_boolean, budget_filter
print('cwd:', os.getcwd(), '| scripts/:', os.path.isdir('scripts'))
print(f'✓ đủ {len(NEED)} module')

In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
# ══ 2Wiki, n=1500, reader 7B ══ ~3.05h
# k=85 lay tu PERCENTILE_BY_DATASET (dung §5.1 paper). Cung MOI cau hinh khac
# giong y lan n=500 - chi doi --limit - de hai lan chay so sanh duoc voi nhau.
import subprocess, sys, os, time
READER='Qwen/Qwen2.5-7B-Instruct'
os.makedirs('results', exist_ok=True)

def run_stream(cmd, logfile):
    t0=time.time()
    with open(logfile,'w') as lf:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}'); return p.returncode

OUT='results/2wiki_1500_7b.json'
if os.path.exists(OUT):
    print(f'[bo qua] da co {OUT}')
else:
    rc=run_stream([sys.executable,'-u','scripts/run_eval.py',
        '--dataset','2wiki','--limit','1500',
        '--reader','hf','--reader-model',READER,'--load-4bit',
        '--itercomp-llm','hf','--scorer','dual',
        '--methods','raw,oracle,llmlingua2,itercomp','--out',OUT],
        'results/log_2wiki_1500.txt')
    assert rc==0, f'2wiki n=1500 that bai (rc={rc})'


In [ ]:
# ══ Khang dinh IterCOMP-Raw da sach chua? Va n=500 co dan minh sai khong? ══
import json, sys, os
sys.path.insert(0, 'repo/src' if os.path.isdir('repo/src') else 'src')
from itercomp.stats import paired_bootstrap

new=json.load(open('results/2wiki_1500_7b.json'))
old=json.load(open('results/2wiki_500_7b.json')) if os.path.exists('results/2wiki_500_7b.json') else None

def cmp(d, tag):
    R=d['per_row']; g=lambda m:[100*r['methods'][m]['f1_norm'] for r in R]
    print(f'  {tag}  (n={len(R)})')
    for a,b in (('itercomp','llmlingua2'),('itercomp','raw'),('oracle','raw')):
        r=paired_bootstrap(g(a),g(b))
        verdict='SACH' if r['lo']*r['hi']>0 else 'cat qua 0'
        print(f"    {a:9s}-{b:11s} {r['diff']:+6.2f} [{r['lo']:+6.2f},{r['hi']:+6.2f}]  {verdict}")

if old: cmp(old,'n=500 (cu)')
cmp(new,'n=1500 (moi)')

# Dau co doi khong? Do la cau hoi "n=500 co dan minh sai" .
gn=lambda d,m:[100*r['methods'][m]['f1_norm'] for r in d['per_row']]
if old:
    a=paired_bootstrap(gn(old,'itercomp'),gn(old,'raw'))
    b=paired_bootstrap(gn(new,'itercomp'),gn(new,'raw'))
    same = (a['diff']>0)==(b['diff']>0)
    print(f"\n  dau IterCOMP-Raw: n=500 {a['diff']:+.2f} -> n=1500 {b['diff']:+.2f}  "
          f"{'GIU NGUYEN' if same else '*** DOI DAU - phai sua bao cao ***'}")
    if b['lo']*b['hi']>0: print('  -> CI da sach: doi "chua ket luan" thanh ket luan that.')
    else: print(f"  -> Van cat 0 o n=1500: hieu that su nho, phat bieu la 'bang 0 trong ±{max(abs(b['lo']),abs(b['hi'])):.1f}'.")


In [ ]:
import shutil, os
BASE='/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z=shutil.make_archive(os.path.join(BASE,'results_2wiki_1500'),'zip','results')
print('✓',z,f'({os.path.getsize(z)/1e6:.1f} MB)')
